# Langfuse Experiment using Ragas Metrics

This notebook demonstrates how to run a [Langfuse Experiment](https://langfuse.com/docs/evaluation/core-concepts#experiments) using [Ragas metrics](https://docs.ragas.io/en/latest/concepts/metrics/overview/). This technique is useful for evaluating RAG quality during CI/CD pipelines to enforce quality thresholds.  

The scoring information is associated with traces and stored in Langfuse, allowing you to further analyze results using [Langfuse dashboards](https://langfuse.com/docs/metrics/features/custom-dashboards) and other capabilities.

**Notes** 
- RAGAS metrics require LLM calls and incur API costs
- Start with small datasets to validate your setup before scaling

## The Pre-Requisites
Before you using this notebook:
- understand the [core concepts](https://langfuse.com/docs/evaluation/core-concepts) of evaluations in Langfuse
- understand the [Ragas metrics](https://docs.ragas.io/en/latest/concepts/metrics/overview/) and their data requirements
- perform the pre-requisites steps outlined in the [README](../README.md).

## The Environment
Establish environment variables and initialize connections as applicable. 

**Recommended:** Take a copy of the `.env.example` (at the root of this repo), save as  `.env`, and update placeholder values in that file.  If you don't do this, you will be prompted to enter the required values during notebook execution.

In [ ]:
from __future__ import annotations

import getpass
import os

from dotenv import load_dotenv

# Load environment variables from .env file (if present)
load_dotenv()


def get_env_or_prompt(var_name: str, prompt_text: str, is_secret: bool = True) -> str:
    """Get environment variable or prompt the user for input.

    Args:
        var_name: Name of the environment variable.
        prompt_text: Text to display when prompting the user.
        is_secret: If True, use getpass to hide input (for API keys).

    Returns:
        The environment variable value or user-provided input.
    """
    value = os.getenv(var_name)
    if value:
        return value
    if is_secret:
        return getpass.getpass(prompt_text)
    return input(prompt_text)


# Langfuse configuration - get from env or prompt
os.environ["LANGFUSE_PUBLIC_KEY"] = get_env_or_prompt(
    "LANGFUSE_PUBLIC_KEY", "Enter Langfuse Public Key: "
)
os.environ["LANGFUSE_SECRET_KEY"] = get_env_or_prompt(
    "LANGFUSE_SECRET_KEY", "Enter Langfuse Secret Key: "
)
os.environ["LANGFUSE_HOST"] = get_env_or_prompt(
    "LANGFUSE_HOST",
    "Enter Langfuse Host URL (default: https://cloud.langfuse.com): ",
    is_secret=False,
) or "https://cloud.langfuse.com"

# OpenAI configuration
os.environ["OPENAI_API_KEY"] = get_env_or_prompt(
    "OPENAI_API_KEY", "Enter OpenAI API Key: "
)

Test Langfuse connection. You should see a success message if the connection is working.

In [ ]:
from langfuse import get_client

# Verify Langfuse connection
langfuse_client = get_client()
if langfuse_client.auth_check():
    print("✅ Langfuse client is authenticated and ready!")
else:
    print("❌ Authentication failed. Please check your credentials and host.")

Initialize Langfuse callback handler.  This will be used by the "task" application, written in LangChain, that generates the traces for evaluation.

In [ ]:
from langfuse.langchain import CallbackHandler  # LangFuse handler

langfuse_handler = CallbackHandler()

## The Dataset
The dataset used for evaluation

For this example, we use the [FIQA (Financial Opinion Mining and Question Answering)](https://huggingface.co/datasets/explodinggradients/fiqa) dataset from Hugging Face. This dataset was prepared by the RAGAS team and contains pre-computed RAG pipeline outputs, making it ideal for demonstrating evaluation workflows without needing a live retrieval system.

The dataset contains the following columns:
- `question`: list[str] - These are the questions your RAG pipeline will be evaluated on.
- `answer`: list[str] - The answer generated from the RAG pipeline and given to the user.
- `contexts`: list[list[str]] - The contexts which were passed into the LLM to answer the question.
- `ground_truths`: list[list[str]] - The ground truth answer to the questions. However, this can be ignored for online evaluations since we will not have access to ground-truth data in our case.

See the [Hugging Face dataset page](https://huggingface.co/datasets/explodinggradients/fiqa) for more details on the dataset structure and provenance.

In [ ]:
from typing import Any, Dict

from datasets import load_dataset

# Load FIQA dataset directly
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")["baseline"].select(
    range(1),
)  # limit to 1 items for testing; remove .select() for full dataset

# Convert to list of dicts for Langfuse experiment
dataset = []
for item in fiqa_eval:
    item_dict: Dict[str, Any] = dict(item)
    dataset.append(
        {
            # input can be structured as needed for the task; here we keep question, contexts, and ground truths together in a dict
            "input": {
                "question": item_dict["question"],
                "contexts": item_dict["contexts"],
                "ground_truths": item_dict["ground_truths"],
            },
            "expected_output": item_dict["answer"],
        },
    )

# print to verify
fiqa_eval

## The LLM Clients
Define the client objects that will communicate with the LLMs.

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize LLM client for Langchain task application
langchain_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


In [ ]:
from openai import AsyncOpenAI
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory

# Initialize LLM client for RAGAS Metrics (the LangChain LLMWrapper was deprecated in RAGAS v0.4+)
async_client = AsyncOpenAI()
llm = llm_factory(
    "gpt-4o-mini", 
    client=async_client,
    max_tokens=8192,
)
# Use embedding_factory (RAGAS v0.4+ recommended pattern from official docs)
# embeddings = embedding_factory("openai", model="text-embedding-3-small", client=async_client)
embeddings = embedding_factory("openai", model="text-embedding-ada-002", client=async_client)

## The Evaluators
Functions to instantiate the Ragas metrics used during Langfuse evaluation

This example includes evaluator functions for the following Ragas metrics:

1. [Faithfulness](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/faithfulness/): This measures the factual consistency of the generated answer against the given context.
2. [AnswerRelevancy](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/answer_relevance/): Answer Relevancy, focuses on assessing how pertinent the generated answer is to the given prompt.
3. [ContextPrecision](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_precision/): Context Precision is a metric that evaluates whether all of the ground-truth relevant items present in the contexts are ranked high. Ideally all the relevant chunks must appear at the top ranks. This metric is computed using the question and the contexts, with values ranging between 0 and 1, where higher scores indicate better precision.

Checkout the [List of available metrics](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/) to know more about these metrics, how they work, and other metrics available for evaluation.

In [ ]:
from typing import cast

from langfuse import Evaluation
from ragas.embeddings.base import BaseRagasEmbedding
from ragas.metrics.collections import AnswerRelevancy, ContextPrecision, Faithfulness

# Initialize metrics with the LLM clients and embeddings
faithfulness = Faithfulness(llm=llm)
answer_relevancy = AnswerRelevancy(
    llm=llm,
    embeddings=cast(BaseRagasEmbedding, embeddings),
    strictness=3,
)
context_precision = ContextPrecision(llm=llm)

# each metric has a unique evaluator function
async def faithfulness_evaluator(*, input, output, expected_output, metadata, **kwargs):
    """Evaluate faithfulness of a response given retrieved contexts."""
    try:
        question = input["question"]
        contexts = input.get("contexts", [])
        result = await faithfulness.ascore(user_input=question, response=output, retrieved_contexts=contexts)
        return Evaluation(name="faithfulness", value=result.value)
    except Exception as e:
        raise RuntimeError(f"RAGAS evaluation failed: {e}") from e

async def answer_relevancy_evaluator(*, input, output, expected_output, metadata, **kwargs):
    """Evaluate answer relevancy of a response given the user input."""
    try:
        question = input["question"]
        result = await answer_relevancy.ascore(user_input=question, response=output)
        return Evaluation(name="answer_relevancy", value=result.value)
    except Exception as e:
        raise RuntimeError(f"RAGAS evaluation failed: {e}") from e

async def context_precision_evaluator(*, input, output, expected_output, metadata, **kwargs):
    """Evaluate context precision of a response given retrieved contexts."""
    try:
        question = input["question"]
        contexts = input.get("contexts", [])
        
        result = await context_precision.ascore(user_input=question, retrieved_contexts=contexts, reference=expected_output)
        return Evaluation(name="context_precision", value=result.value)
    except Exception as e:
        raise RuntimeError(f"RAGAS evaluation failed: {e}") from e

# run-level evaluators that calculate averages across all evaluation items for the entire experiment
def average_evaluator(metric_name, item_results):
    """Run evaluator that calculates average across all items for the specified metric"""
    scores = [
        eval.value for result in item_results
        for eval in result.evaluations if eval.name == metric_name
    ]
    if not scores:
        return Evaluation(name=f"avg_{metric_name}", value="n/a", comment=f"No scores found for metric '{metric_name}'")
    avg = sum(scores) / len(scores)
    return Evaluation(name=f"avg_{metric_name}", value=avg, comment=f"Average {metric_name}: {avg:.2%}")

def average_faithfulness_evaluator(*, item_results, **kwargs):
    """Run evaluator that calculates average faithfulness across all items"""
    return average_evaluator("faithfulness", item_results)

def average_answer_relevancy_evaluator(*, item_results, **kwargs):
    """Run evaluator that calculates average answer relevancy across all items"""
    return average_evaluator("answer_relevancy", item_results)

def average_context_precision_evaluator(*, item_results, **kwargs):
    """Run evaluator that calculates average context precision across all items"""
    return average_evaluator("context_precision", item_results)

## The Task
Simple LangChain task that takes in the questions and contexts from the dataset then queries the LLM for answers.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# define template that we will "stuff" with questions and context during task execution
prompt = ChatPromptTemplate.from_template("""
Answer based ONLY on the following contexts:
{contexts}

Question: {question}
""")

def langchain_task(*, item, **kwargs):
  # parse dataset item
  question = item['input'].get("question", "")
  contexts = "\n".join(item['input'].get("contexts", []))
  # Build chain: prompt → LLM
  chain = prompt | langchain_llm
  # Invoke chain, injecting context and question directly, and pass Langfuse callback handler in config to capture trace
  result = chain.invoke(
      {"contexts": contexts, "question": question},
      config={
        "callbacks": [langfuse_handler], 
      }
  )
  return result.content

## The Experiment
Conduct the Langfuse experiment using the task and evaluators defined in previous sections.

In [ ]:
result = langfuse_client.run_experiment(
    name="Ragas Metrics Evaluation",
    description="Evaluate RAGAS metrics on FIQA dataset using Langfuse and Langchain",
    data=dataset,  # dataset to loop over in the experiment
    task=langchain_task, # the task to run for each item in the dataset
    # item-level evaluators that score the output of the task for each item in the dataset
    evaluators=[faithfulness_evaluator, answer_relevancy_evaluator, context_precision_evaluator],  
    # run-level evaluators that calculate averages across all evaluation items
    run_evaluators=[average_faithfulness_evaluator, average_answer_relevancy_evaluator, average_context_precision_evaluator],
)

You can examine the average scores for each metric to enforce quality thresholds.

In [ ]:
 
# Access the run evaluator result directly
avg_faithfulness = next(
    eval.value for eval in result.run_evaluations
    if eval.name == "avg_faithfulness"
)
print(f"Average Faithfulness: {avg_faithfulness:.2%}")
avg_relevancy = next(
    eval.value for eval in result.run_evaluations
    if eval.name == "avg_answer_relevancy"
)
print(f"Average Answer Relevancy: {avg_relevancy:.2%}")
avg_context_precision = next(
    eval.value for eval in result.run_evaluations
    if eval.name == "avg_context_precision"
)
print(f"Average Context Precision: {avg_context_precision:.2%}")